In [ ]:
#import libraries
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import nltk
import re
import spacy
import sys
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import word_tokenize
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.feature_extraction.text import TfidfVectorizer
import string
import warnings
warnings.filterwarnings('ignore')

nltk.download('wordnet')
nltk.download('omw-1.4')


nltk.download('stopwords')

!python -m spacy download en_core_web_sm
!python -m spacy download xx_ent_wiki_sm

#setting configuration for all our plots
sns.set_style('darkgrid')
sns.set_palette('Set2')
plt.rcParams['figure.figsize'] = (8,5)

In [ ]:
!pip install langid

In [ ]:
#from google.colab import files
#uploaded = files.upload()

In [ ]:
data_path = Path("/content/drive/MyDrive/raw_reviews.csv")
df = pd.read_csv(data_path)
df.head()

In [ ]:
#df = pd.read_csv("raw_reviews.csv")
#df.head()

In [ ]:
sample_df = df.sample(n=100, random_state=42)
sample_df.head()
sample_df.to_csv("/content/drive/MyDrive/sample_reviews.csv", index=False)

In [ ]:
df.shape

In [ ]:
df.duplicated().sum()

In [ ]:
df.isna().sum()

In [ ]:
df.info()

In [ ]:
df.columns

EXPLORATORY DATA ANALYSIS

In [ ]:
df.product_category.value_counts().reset_index()

In [ ]:
df['product_category'].value_counts().sort_index().plot(kind='bar', title='Product Distribution')
plt.xlabel('Product Category')
plt.ylabel('Count')
plt.show()

From the plot we can see that Product categories are evenly distributed (no major imbalance).
Ensures fair comparison across categories in analysis.

Rating Distribution per product category

In [ ]:
#plot of rating per product_category
df.groupby('product_category')['rating'].count().reset_index()

In [ ]:
# Rating Distribution by Product Category
sns.countplot(y='product_category', hue='rating', data=df)
plt.title('Rating Distribution by Product Category')
plt.xlabel('Count')
plt.ylabel('Product Category')
plt.legend(loc='lower left', bbox_to_anchor=(-0.2, 0.6))
plt.show()

This plot shows that all categories show a similar pattern:
High 5-star dominance followed by 4-star ratings. No category shows unusually poor ratings.

In [ ]:
df.groupby('product_category')['sentiment'].count().reset_index()

In [ ]:
# Sentiment Distribution by Product Category
sns.countplot(y='product_category', hue='sentiment', data=df)
plt.title('Sentiment Distribution by Product Category')
plt.xlabel('Count')
plt.ylabel('Product Category')
plt.legend(loc='lower right', bbox_to_anchor=(-0.2, 0.1))
plt.show()

This plot shows that categories are dominated by positive sentiment.
Slight variations exist, but no category stands out as highly negative.


Do “positive” reviews actually have high ratings?

In [ ]:
#Average rating per sentiment (per category)
df.groupby(['product_category', 'sentiment'])['rating'].mean().reset_index()

Do customers rate products consistently with their written sentiment?

In [ ]:
#Checking for mismatch records of product ratings and sentiment
mismatch = df[
    ((df['rating'] >= 4) & (df['sentiment'] == 'negative')) |
    ((df['rating'] <= 2) & (df['sentiment'] == 'positive'))
]
mismatch.head()

In [ ]:
#Checking for inconsitencies in rating and sentiment
sns.boxplot(data=df, x='sentiment', y='rating')
plt.xticks(rotation=45)
plt.show()

Across product categories, sentiment labels generally align with customer ratings, with positive reviews showing higher average ratings. However, certain categories exhibit inconsistencies, where negative sentiment appears alongside high ratings. This suggests either customer experience nuances or potential misclassification in sentiment labeling, highlighting areas for further investigation.


Rating Distributing

In [ ]:
# checking for rating distribution to know the emotions of our customers
df['rating'].value_counts().reset_index()

In [ ]:
df['rating'].value_counts().sort_index().plot(kind='bar', title='Rating Distribution')
plt.xlabel('Rating')
plt.ylabel('Count')
plt.show()

The plot is clearly skewwed toward higher ratings (4 and 5 stars).
5-star ratings are the most frequent, followed by 4-star.
Very few low ratings

Reviews by Country

In [ ]:
df['country'].value_counts().reset_index()

In [ ]:
# Checking the country with the highest review
sns.countplot(x='country', data=df)
plt.xticks(rotation=90)
plt.title('Reviews by Country')
plt.show()

Highest activity comes from US, GB, and DE.
Other countries contribute smaller but consistent volumes.


Sentiment Distribution per Country

In [ ]:
df['sentiment'].value_counts().reset_index()

In [ ]:
# Checking the country with the highest Sentiment count
sns.countplot(y='country', hue = 'sentiment', data=df)
plt.xticks(rotation=90)
plt.title('Sentiment per country')
plt.legend()
plt.show()

Positive sentiment dominates across all countries.
US, GB, and DE again lead in volume of positive reviews.
Neutral and negative sentiments are significantly lower, showing consistency with high ratings.

Rating Distribution per country

In [ ]:
# Checking the country with the highest Rating count
sns.countplot(y='country', hue = 'rating', data=df)
plt.xticks(rotation=90)
plt.title('Ratings per Country')
plt.legend()
plt.show()

Most countries show a strong dominance of 4 and 5-star ratings.
US, GB, and DE have the highest review volumes and highest 5-star counts.
Lower ratings (1–2 stars) are relatively rare across all countries.
Suggesting generally high customer satisfaction globally.

Spotting the Noise
The noise in the review column includes emoji, uppercase, symbols etc

In [ ]:
def has_numbers(text):
    return bool(re.search(r'\d', str(text)))

def has_emoji(text):
    return bool(re.search(r'[\U00010000-\U0010ffff]', str(text)))

def is_uppercase(text):
    return str(text).isupper()

def has_symbols(text):
    return bool(re.search(r'[^\w\s]', str(text)))  # punctuation/symbols

def has_multilingual(text):
    # Detect non-English characters (basic approach)
    return bool(re.search(r'[^\x00-\x7F]', str(text)))

In [ ]:
def sample_reviews(df, condition_func, n=5):
    subset = df[df['review'].apply(condition_func)]
    return subset['review'].sample(min(n, len(subset)), random_state=42)

In [ ]:
print("Reviews with numbers:")
print(sample_reviews(df, has_numbers), "\n")

print("Reviews with emojis:")
print(sample_reviews(df, has_emoji), "\n")

print("Uppercase reviews:")
print(sample_reviews(df, is_uppercase), "\n")

print("Reviews with symbols/punctuation:")
print(sample_reviews(df, has_symbols), "\n")

print("Multilingual reviews:")
print(sample_reviews(df, has_multilingual), "\n")

Exploratory text analysis revealed the presence of noise such as emojis, multilingual content, numeric-heavy reviews, and excessive punctuation. These patterns informed preprocessing decisions including text normalization, token cleaning, and potential language filtering to improve model performance.

DATA CLEANING

In [ ]:
#Fix date type
df['timestamp'] = pd.to_datetime(df['timestamp'], errors='coerce')
df['rating'] = pd.to_numeric(df['rating'], errors='coerce')

In [ ]:
#checking for dataset info again
df.info()

In [ ]:
#Standardize text (Review column)
df['review'] = df['review'].str.lower()  # lowercase
df['review'] = df['review'].str.strip()  # remove spaces

In [ ]:
#Removing noise from text

def clean_text(text):
    text = re.sub(r'http\S+', '', text)              # remove URLs
    text = re.sub(r'@\w+', '', text)                 # remove mentions
    text = re.sub(r'\d+', '', text)                  # Remove numbers
    text = re.sub(r'[^a-zA-Z\s]', '', text)          # remove symbols
    text = re.sub(r'[\U00010000-\U0010ffff]', '', text)  # remove emojis
    text = re.sub(r'[^\w\s]', '', text)               # remove punctuation
    return text

df['clean_review'] = df['review'].apply(clean_text)

In [ ]:
#remove extra white space
df['clean_review'] = df['clean_review'].apply(
    lambda x: re.sub(r'\s+', ' ', str(x)).strip()
)

In [ ]:
df[['review', 'clean_review']].sample(10)

In [ ]:
# functions to check individual rows for messiness
def has_emoji(text):
    return bool(re.search(r'[\U00010000-\U0010ffff]', str(text)))

def has_numbers(text):
    return bool(re.search(r'\d', str(text)))

def has_uppercase(text):
    return any(char.isupper() for char in str(text))

def has_extra_whitespace(text):
    return bool(re.search(r'\s{2,}', str(text)))

def has_punctuation(text):
    return bool(re.search(r'[^\w\s]', str(text)))

In [ ]:
#checking if rows have been properly cleaned
emoji_count = df['clean_review'].apply(has_emoji).sum()
number_count = df['clean_review'].apply(has_numbers).sum()
uppercase_count = df['clean_review'].apply(has_uppercase).sum()
whitespace_count = df['clean_review'].apply(has_extra_whitespace).sum()
punctuation_count = df['clean_review'].apply(has_punctuation).sum()

print("Emoji rows:", emoji_count)
print("Number rows:", number_count)
print("Uppercase rows:", uppercase_count)
print("Whitespace rows:", whitespace_count)
print("Punctuation rows:", punctuation_count)

In [ ]:
#Reviewing processed data (clean_review) after cleaning
df['clean_review'].head(10)

During the data cleaning stage, I converted text to lowercase.
Removed punctuation, numbers, emojis, special characters and extra whitespace

The purpose of the data cleaning is to standardize text format, reduce noise that does not contribute to sentiment meaning and ensure consistent input for NLP models

The impact of the data cleaning is to eliminated irrelevant characters that could confuse the model, improved token consistency (e.g., “Good”, “good”, “GOOD” → “good”) and reduced feature dimensionality during vectorization

Lemmatization

In [ ]:
import langid

def possibly_multilingual(text):
    return bool(re.search(r'[^\x00-\x7F]', str(text)))

def detect_language_fast(text):
    try:
        return langid.classify(str(text)[:200])[0]
    except:
        return "unknown"

df['language'] = 'en'
mask = df['review'].apply(possibly_multilingual)
df.loc[mask, 'language'] = df.loc[mask, 'review'].apply(detect_language_fast)

To reduce computational cost and time, I used a faster language detection (langid) which was applied selectively (only on the first 200 characters) to reviews likely to contain non-English content, while English was assumed as the default for the remaining reviews.

In [ ]:
#checks for all the language used in the review
df['language'].unique()

In [ ]:
#checks for the column in the dataset
print(df.columns)

In [ ]:
#lemmatize text function
nlp_en = spacy.load("en_core_web_sm", disable=["parser", "ner", "textcat"])
nlp_multi = spacy.load("xx_ent_wiki_sm", disable=["parser", "ner", "textcat"])

def lemmatize_multilingual(text, lang):
    text = str(text)

    if lang == 'en':
        doc = nlp_en(text)
    else:
        doc = nlp_multi(text)

    tokens = [token.lemma_ if token.lemma_ else token.text for token in doc]
    return " ".join(tokens).strip()

df['lemma_text_review'] = df.apply(
    lambda row: lemmatize_multilingual(row['clean_review'], row['language']),
    axis=1
)
df.head(10)

Lemmatization was used to reduce words to their base/root form(lemma).
Applied primarily to English and multilingual reviews using spaCy.

The purpose is to normalize word variations and ensure similar words are treated as the same feature

Lemmatization helped to reduce vocabulary size, improve generalization of the model and helped group similar meanings.


In [ ]:
# check for column after lemmatization
df.columns

Removal of stopwords

In [ ]:
stop_words = set(stopwords.words('english'))

df['final_review'] = df['lemma_text_review'].apply(
    lambda x: ' '.join([word for word in x.split() if word not in stop_words])
)
df.head(10)

Stop word removal helped to remove common words by applying language-specific stopword filtering.

The purpose is to remove non-informative words, focus on words that carry sentiment meaning.

The impact of this to the dataset is to reduce noise in text representation, improve model efficiency and performance.


In [ ]:
#creating label column to group rating into 0 for (1,2), 1 for (3) and 2 for (4,5)
df['label'] = df['rating'].apply(lambda r: 0 if r in (1,2) else ( 1 if r == 3 else 2))
df.head()

In [ ]:
# Save the df as clean csv
df.to_csv("/content/drive/MyDrive/clean_review.csv", index=False)

In [ ]:
#Review all derived columns and their processed data
df[['review', 'clean_review', 'lemma_text_review', 'final_review']].head(10)

After preprocessing, the dataset shows no presence of emojis, numbers, uppercase inconsistencies, or punctuation noise. Remaining whitespace reflects normal word separation rather than data quality issues. Additional checks confirmed minimal presence of excessive or irregular spacing.”

BASELINE MODELING

In [ ]:
sentiment_data = df.copy()
sentiment_data.head()

In [ ]:
#creating features to be used by the model by indexing three selected columns from the full sentiment dataset
sentiment_data = sentiment_data[['review','final_review','label']]
sentiment_data.head()

In [ ]:
sentiment_data.shape

In [ ]:
sentiment_data['label'].value_counts().reset_index()

In [ ]:
sentiment_data.info()

In [ ]:
# Importing model libraries
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix


In [ ]:
X = sentiment_data['final_review'].astype(str)
y = sentiment_data['label']

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

In [ ]:
vectorizer = TfidfVectorizer(max_features=5000)
X_train_num = vectorizer.fit_transform(X_train)


In [ ]:
lr_model = LogisticRegression()
lr_model.fit(X_train_num, y_train)

In [ ]:
X_test_num = vectorizer.transform(X_test)

In [ ]:
lr_model.score(X_test_num, y_test)

In [ ]:
classification_report = classification_report(y_test, lr_model.predict(X_test_num))
print("Classification Report:")
print(classification_report)

In [ ]:
y_pred = lr_model.predict(X_test_num)
conf_matrix = confusion_matrix(y_test, y_pred)
print("Confusion Matrix:")
print(conf_matrix)

In [ ]:
#plot for confusion matrix
from sklearn.metrics import ConfusionMatrixDisplay

labels = ['negative', 'neutral', 'positive']
disp = ConfusionMatrixDisplay(confusion_matrix=conf_matrix, display_labels=labels)
disp.plot(cmap=plt.cm.Blues)

plt.title("Confusion Matrix")
plt.show()

From the confusion matrix, the correct predictions are:
- Negative → Negative: 3575
- Neutral → Neutral: 3505
- Positive → Positive: 16108

Negative Class: Correct (True) is 3575, while misclassified (False) is Neutral: 51, Positive: 51. This gives very few errors

Neutral Class: Correct classification (True) is 3505 while misclassified (False) is Negative: 59, Positive: 60. Slight confusion with both sides.

Positive Class: Correct classification(True) is 16108 (very high) while misclassified (False) is Negative: 314, Neutral: 277.
Positive class is much larger. So small % error is acceptable.

The confusion matrix shows that the dataset is imbalanced. The model demonstrates particularly high accuracy for positive reviews (16k vs ~3.5k others), reflecting the dominant class distribution. Minor misclassifications occur between neutral and adjacent sentiment classes, which is expected due to their semantic similarity. The confusion matrix shows strong model performance, with the majority of predictions correctly classified across all sentiment categories. Overall, the model exhibits robust and reliable sentiment classification performance.

This is the baseline model using TfidfVectorizer as the text vectorizer and logistic regression to fit the model. The Logistic Regression model achieved strong performance with 97% accuracy and a weighted F1-score of 0.97. The model demonstrated high precision and recall across all sentiment classes, with particularly strong performance in identifying positive reviews. Minor misclassifications were observed between neutral and adjacent sentiment classes, which is expected due to their semantic overlap. Despite class imbalance which acts as a limitation to the baseline model, the model maintained balanced performance, indicating robust generalization.


BERT THEORY

BERT (Bidirectional Encoder Representations from Transformers) is a deep learning model developed by Google for understanding text. It is a transformer-based model that understands text context bidirectionally. It uses subword tokenization to break words into meaningful units and applies self-attention to capture relationships between all words in a sentence.

The core idea of BERT is that it reads text in both directions at the same time. Reads left + right together (bidirectional).

Key Concepts in BERT
- Transformers Architecture: BERT is based on the Transformer architecture. Uses self-attention

- Self-Attention: Every word looks at every other word in the sentence. Example: "I love this product because it is amazing". “it” understands it refers to “product”

- Pre-training Tasks
BERT learns language using two main tasks:

- Masked Language Modeling (MLM)
"I love this [MASK]". Model predicts missing word → “product”

-  Next Sentence Prediction (NSP): Learns relationship between sentences

How Tokenization Works in BERT:

- Step 1: Split text into tokens. BERT does NOT use simple word splitting. It uses WordPiece Tokenization. WordPiece Tokenization breaks words into smaller subwords. E.g: "playing" →["play", "##ing"], "unhappy" → ["un", "##happy"],
 where “##” means continuation of a word

- Step 2: Add Special Tokens: BERT adds:
[CLS] → start of sentence, [SEP] → end of sentence. Example: "I love NLP" becomes:
[CLS] I love NLP [SEP].

- Step 3: Convert tokens to IDs. Each token → numeric ID. "I" → 1045, "love" → 2293. Model only understands numbers.

- Step 4: Attention Masks
Tells model which tokens are real vs padding.
Tokenization:
[CLS], I, am, play, ##ing, football, [SEP].


In [ ]:
from transformers import AutoTokenizer
#Load the BERT Tokenizer
model_name = "distilbert-base-multilingual-cased"
tokenizer = AutoTokenizer.from_pretrained(model_name)


Tokenization

In [ ]:
# Convert pandas Series to list of strings
X_train = X_train.tolist() if not isinstance(X_train, list) else X_train
X_test = X_test.tolist() if not isinstance(X_test, list) else X_test

In [ ]:
from transformers import BertTokenizer
#Load the BERT Tokenizer
model_name = "distilbert-base-multilingual-cased"
tokenizer = BertTokenizer.from_pretrained(model_name)



In [ ]:
#inspecting X_train before tokenization
print(type(X_train))
print(list(X_train))

In [ ]:
# Tokenize texts
train_encodings = tokenizer(X_train, truncation=True, padding=True, max_length=128)
test_encodings = tokenizer(X_test, truncation=True, padding=True, max_length=128)

Convert to Pytorch Dataset

In [ ]:
import torch

class SentimentDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        #Ensuring labels are list to avoid pandas index issues
        if hasattr(labels, 'tolist'):
            self.labels = labels.tolist() # Convert pandas series to list
        elif hasattr(labels, '__iter__') and not isinstance(labels, (list, tuple)):
            self.labels = list(labels)
        else:
            self.labels = labels
    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx])
        return item

# Converting labels to list to avoid pandas index issues
if hasattr(y_train, 'tolist'):
    y_train = y_train.tolist()
if hasattr(y_test, 'tolist'):
    y_test = y_test.tolist()

# Create dataset
train_dataset = SentimentDataset(train_encodings, y_train)
test_dataset = SentimentDataset(test_encodings, y_test)

Loading Pre-trained Bert for Classification

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=3)

In [ ]:
import transformers
print(transformers.__version__)


In [ ]:
from transformers import TrainingArguments, Trainer

training_args = TrainingArguments(
    output_dir='./results',
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_dir="./logs",
    logging_steps=50,
    save_total_limit=1,
    load_best_model_at_end=True,
    metric_for_best_model="accuracy"
)

Define Metrics

In [ ]:
import numpy as np
from sklearn.metrics import accuracy_score, f1_score

def compute_metrics(p):
    preds = np.argmax(p.predictions, axis=1)
    labels = p.label_ids

    accuracy = accuracy_score(labels, preds)
    f1 = f1_score(labels, preds, average='weighted')
    return {"accuracy": accuracy, "f1": f1}

Train the Model

In [ ]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics
)

trainer.train()

Evaluate

In [ ]:
results = trainer.evaluate()
print(results)

Make Predictions

In [ ]:
preds = trainer.predict(test_dataset)
bert_preds = np.argmax(preds.predictions, axis=1)

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report

conf_matrix = confusion_matrix(y_test, bert_preds)
print("Confusion Matrix:")
print(conf_matrix)


In [ ]:
report = classification_report(y_test, bert_preds)
print("Classification Report:")
print(report)


After the getting the reults from the confusion matrix and classification report, we noticed that the results are same as our traditional machine learning model, Logistic regression which is our baseline model. The only reason we are choosing Bert over the traditional machine learning model is because of Bert's capabilities to be able to know sentence/text predictions and understand context.

In [ ]:
#Plot the comfusion matrix
labels = ['negative', 'neutral', 'positive']
disp = ConfusionMatrixDisplay(confusion_matrix=conf_matrix, display_labels=labels)
disp.plot(cmap=plt.cm.Blues)

plt.title("Confusion Matrix")
plt.show()

Testing the model after training

In [ ]:
print("\n" + "="*50)
print("TESTING THE MODEL")
print("="*50)

#function to test single prediction
def test_single_prediction(text, model, tokenizer):
    # Tokenize
    inputs = tokenizer(text, padding=True, truncation=True, return_tensors="pt", max_length=128)

    #move to same device as model
    device = next(model.parameters()).device
    inputs = {key: val.to(device) for key, val in inputs.items()}

    # Make prediction
    model.eval()
    with torch.no_grad():
        outputs = model(**inputs)
        probabilities = torch.nn.functional.softmax(outputs.logits, dim=1)
        prediction = torch.argmax(probabilities, dim=1).item()

    # Map to label(Adjust based on label mapping)
    id2label = {0: 'negative', 1: 'neutral', 2: 'positive'}
    predicted_label = id2label[prediction]
    confidence = probabilities[0][prediction].item()

    return predicted_label, confidence, probabilities[0].cpu().numpy()

#Testing on sample text
test_sample = [
    "I am very impressed with this product",
    "solid build attractive design works as seen in advert",
    "am highly disappointed by the package",
	"three stars meets the minimum expectations",
    "Amazing! I would highly recommend",
"broken on arrival return process was a nightmare",
"terrible delivery services",
"superb quality very sturdy and wellmade five",
"its okay, nothing special"
]

print("/testing on sample test:")
print("-" * 70)
for text in test_sample:
    predicted_label, confidence, probs = test_single_prediction(text, model, tokenizer)
    print(f"Text: {text}")
    print(f"Sentiment: {predicted_label}")
    print(f"Confidence: {confidence:.3f}")
    print(f"Probabilities: negative={probs[0]:.3f}, neutral={probs[1]:.3f}, positive={probs[2]:.3f}")
    print("-" * 70)

Model Comparison: Logistic Regression vs BERT Overall Performance
- Metric	  Logistic Regression	  BERT
- Accuracy	0.97	                0.97
- Macro F1	0.95	                0.95
- Weighted F1	0.97	              0.97

Result:Both models achieved identical overall performance

Class-Level Comparison

Negative (Class 0)
1. Precision: 0.91
2. Recall: 0.97
3. F1: 0.94

- Both models have strong recall which captures most negative reviews,
slight precision drop (some false positives)

Neutral (Class 1)
1. Precision: 0.91
2. Recall: 0.97
3. F1: 0.94

- Both models Perform similarly


Positive (Class 2)
1. Precision: 0.99
2. Recall: 0.96
3. F1: 0.98

- Both models are extremely strong, very confident predictions and
Confusion Matrix Insight is same for both.

Key Insight

BERT did NOT outperform Logistic Regression

1. Dataset is “easy” to classify
Strong sentiment words
Clear patterns
Minimal uncertainty
2. TF-IDF + Logistic Regression is already powerful
Works very well for sentiment tasks
Captures key word signals effectively
3. BERT advantage not fully needed

BERT excels when:

context is complex
sarcasm or nuance exists
long dependencies matter

The dataset likely does NOT require the level of complexity of Bert

While BERT provides advanced contextual understanding of customer reviews, its performance in this project was comparable to the Logistic Regression baseline. Given the significantly higher computational cost and complexity, BERT does not currently offer additional business value for ShopEase. However, it remains a strong candidate for future implementation as the platform scales or if customer feedback becomes more complex.

In [ ]:
#After training, save the model and tokenizer
model_save_path = "/content/drive/MyDrive/Sentiment_Analysis_for_ShopEase"
# save model
model.save_pretrained(model_save_path)
# save tokenizer
tokenizer.save_pretrained(model_save_path)

print(f"Model saved to {model_save_path}")

In [ ]:
!pip install dagshub mlflow

In [ ]:
import dagshub
import mlflow
from transformers import pipeline
from mlflow.tracking import MlflowClient
# Evaluate
metrics = trainer.evaluate(eval_dataset=test_dataset)

sentiment_pipeline = pipeline(
    task="text-classification",
    model=model,
    tokenizer=tokenizer,
    return_all_scores=True
)

dagshub.init(repo_owner='isiakpereaghogho',
             repo_name='Sentiment_Analysis_for_ShopEase_Customer_Feedback',
             mlflow=True
)

mlflow.set_experiment("sentiment_analysis_experiment")

#log model and metrics
with mlflow.start_run():
  #log model to MLflow and register
  mlflow.transformers.log_model(
      transformers_model=sentiment_pipeline,
      name="model",
      registered_model_name="distilbert_sentiment_model"
  )

  #log evaluation metrics

  mlflow.log_metric("accuracy", metrics["eval_accuracy"])
  mlflow.log_metric("f1", metrics["eval_f1"])

  mlflow.log_metric("eval_accuracy", metrics["eval_accuracy"])
  mlflow.log_metric("eval_f1", metrics["eval_f1"])

  if "eval_loss" in metrics:
    mlflow.log_metric("eval_loss", metrics["eval_loss"])

  # mlflow.log_metric("eval_accuracy",metrics["eval_accuracy"])
  # mlflow.log_metric("eval_f1",metrics["eval_f1"])
  # mlflow.log_metric("eval_loss",metrics["eval_loss"])

  #log training parameter
  mlflow.log_param("num_epoch",training_args.num_train_epochs)
  mlflow.log_param("batch_size",training_args.per_device_train_batch_size)
  mlflow.log_param("learning_rate",training_args.learning_rate)

  print("Model successfully pushed to MLflow!")

  # verify it's registered
  print("\nCheck registered model:")

  client = MlflowClient()

  for mv in client.search_model_versions("name='distilbert_sentiment_model'"):
    print(f"Version: {mv.version}, Stage: {mv.current_stage}, Run ID: {mv.run_id}")